# Q4 — Billed vs Paid Ratio

## Business Question

How much of the amount providers bill does the insurer actually pay, and does this vary by claim type, provider, or CPT code?

## Objective

Calculate the ratio between the amount billed by providers and the amount paid by the insurer. Compare this ratio across claim types, providers, and CPT codes to identify significant differences between billed and paid amounts.

In [1]:
import pandas as pd

In [2]:
claims = pd.read_csv('/Users/blessingmashonga/Downloads/healthcare-claims-where-is-the-money-going-tvhlq/claims.csv')

## Q4A — Overall Paid Ratio

The paid ratio is calculated as:

Paid Ratio = Total Paid Amount / Total Billed Amount

Using total amounts provides a weighted ratio because larger claims have a greater effect on the result.

In [3]:
overall_paid_ratio = (
    claims['paid_amount'].sum() /
    claims['billed_amount'].sum()
)

overall_paid_ratio

np.float64(0.7519984445473946)

## Q4B — Paid Ratio by Claim Type

Compare the paid ratio across claim types to determine whether the proportion of billed amounts paid by the insurer varies across different types of healthcare services.

In [4]:
claim_type_expense = (
    claims.groupby('claim_type')
    .agg({
        'paid_amount': 'sum',
        'billed_amount': 'sum',
        'claim_id': 'count'
    })
    .reset_index()
)

claim_type_expense['paid_ratio'] = (
    claim_type_expense['paid_amount'] /
    claim_type_expense['billed_amount']
)

claim_type_expense = claim_type_expense.sort_values(
    'paid_ratio',
    ascending=False
)

claim_type_expense

,claim_type,paid_amount,billed_amount,claim_id,paid_ratio
2,Lab,23412.35,25789.90,76,0.907811
4,Pharmacy,11382.45,12814.60,81,0.888241
3,Outpatient,129052.75,160717.75,105,0.802978
0,Emergency,294441.36,384241.55,88,0.766292
1,Inpatient,1092456.00,1478601.25,99,0.738844


## Q4C — Paid Ratio by Provider

Compare paid ratios across providers to identify providers with unusually high or low ratios.

Provider-level ratios should be interpreted alongside financial magnitude and claim volume.

In [5]:
provider_ratio = (
    claims.groupby('provider_id')
    .agg({
        'paid_amount': 'sum',
        'billed_amount': 'sum',
        'claim_id': 'count'
    })
    .reset_index()
)

provider_ratio['paid_ratio'] = (
    provider_ratio['paid_amount'] /
    provider_ratio['billed_amount']
)

provider_ratio.sort_values(
    'paid_ratio',
    ascending=True
).head(10)

,provider_id,paid_amount,billed_amount,claim_id,paid_ratio
10,PRV00214,0.0,210.0,1,0.000000
14,PRV00456,9800.0,27750.0,2,0.353153
60,PRV08910,5120.0,12850.0,1,0.398444
43,PRV03422,140.0,310.0,1,0.451613
9,PRV00156,9000.0,14500.0,1,0.620690
50,PRV05432,3550.0,5400.0,1,0.657407
0,PRV00001,151304.4,228729.0,39,0.661501
100,PRV98765,21700.0,32795.0,13,0.661686
49,PRV04789,14000.0,21000.0,1,0.666667
61,PRV09012,3600.0,5400.0,1,0.666667


## Q4D — CPT Paid Ratio

Compare paid ratios across CPT codes to identify procedures with large differences between billed and paid amounts.

In [6]:
cpt_ratio = (
    claims.groupby('cpt_code')
    .agg({
        'paid_amount': 'sum',
        'billed_amount': 'sum',
        'claim_id': 'count'
    })
    .reset_index()
)

cpt_ratio['paid_ratio'] = (
    cpt_ratio['paid_amount'] /
    cpt_ratio['billed_amount']
)

cpt_ratio.sort_values(
    'paid_ratio',
    ascending=True
).head(10)

,cpt_code,paid_amount,billed_amount,claim_id,paid_ratio
10,10001,4000.0,42000.0,1,0.095238
33,27650,5120.0,12850.0,1,0.398444
68,52345,2700.0,4500.0,1,0.600000
58,456,6050.0,9750.0,5,0.620513
120,99220,9800.0,15000.0,1,0.653333
119,99217,10200.0,15200.0,1,0.671053
60,460,9800.0,14500.0,1,0.675862
106,93000,20650.0,30210.0,10,0.683548
3,0009F,65.0,95.0,1,0.684211
79,66789,3600.0,5200.0,1,0.692308


## Q4E — Investigating Significant Differences

Extreme paid ratios can identify claims or providers that warrant further investigation.

However, a low ratio does not necessarily indicate a major financial impact. The amount billed, amount paid, and number of claims should also be considered.

In [8]:
cpt_10001 = claims[claims['cpt_code'] == '10001']

cpt_10001

,claim_id,member_id,provider_id,claim_date,claim_type,cpt_code,icd_code,billed_amount,paid_amount
13,14,4,PRV00001,03-15-2023,Inpatient,10001,A00.0,42000.0,4000.0


In [9]:
cpt_10001[['billed_amount', 'paid_amount']].sum()

billed_amount    42000.0
paid_amount       4000.0
dtype: float64

## Q4 Summary

Paid ratio provides a useful measure of the relationship between provider-billed amounts and insurer-paid amounts.

However, paid ratio alone does not identify healthcare cost drivers. A claim with a low paid ratio can still have a substantially higher paid amount than a claim with a 100% paid ratio.

Therefore, differences between billed and paid amounts should be considered alongside the financial magnitude and volume of claims. Further investigation would be required to determine why specific claims were not paid in full, as the dataset does not contain detailed adjustment or denial reasons.